# Sesión 2 · Aprender con etiqueta
**Machine Learning y Seguridad en la Nube — Especialización en Big Data (UCompensar)**

Este notebook acompaña las [diapositivas de la sesión](https://jotamao1985.github.io/Machine-Learning-cloud-computing_Ucompensar/diapositivas/sesion-2-aprender-con-etiqueta.html).
Cada celda lleva entre corchetes el mismo identificador que el código proyectado, así que puedes seguir
la clase celda a celda. La portada del curso está en https://jotamao1985.github.io/Machine-Learning-cloud-computing_Ucompensar/
y el enunciado del proyecto final, en [su página](https://jotamao1985.github.io/Machine-Learning-cloud-computing_Ucompensar/proyecto.html).

1. **Archivo → Guardar una copia en Drive** para que tus cambios no se pierdan.
2. Ejecuta las celdas **en orden** con **Mayús + Enter**. Varias usan resultados de las anteriores.
   Si Colab avisa que el notebook no es de Google, elige **Ejecutar de todos modos**.
3. Si Colab se reinicia o una celda falla al repetirla, usa **Entorno de ejecución → Ejecutar anteriores**.
   Las celdas de ejercicio dan error mientras tengan `...` sin reemplazar (si el error menciona `Ellipsis`,
   te quedó un `...`). Colab se detiene en el primer error: completa los ejercicios en orden, o ejecuta
   con **Mayús + Enter** solo las celdas que necesites.
4. La primera celda **descarga** los datos del caso *Yarumo Fibra S.A.S.*, una empresa **ficticia** (sus
   datos se generaron para el curso y no corresponden a personas ni a empresas reales), y comprueba que
   el archivo sea idéntico al del curso. Si la descarga falla, la celda te dice cómo subirlo a mano.

En el código, los decimales van con punto (así los escribe Python); en el texto, con coma.

In [ ]:
#@title ▶ Descarga los datos del caso (ejecútala sin modificar) { display-mode: "form" }
"""Datos del caso «Yarumo Fibra S.A.S.» (empresa FICTICIA), descargados del repositorio del curso.

Deja en datos/clientes.csv los 8000 clientes del curso y comprueba su huella SHA-256: si el
archivo no es idéntico al del curso, no se usa. Solo usa la biblioteca estándar de Python.
"""

import hashlib
import os
import urllib.request
from pathlib import Path

DATOS_URL = "https://raw.githubusercontent.com/JotaMao1985/Machine-Learning-cloud-computing_Ucompensar/main/datos/clientes.csv"
DATOS_SHA256 = "2d4fad61a49efcf1d071b6c412e5224e71eca23047cbbac6b4017c8627f9c93b"
PORTADA = "https://jotamao1985.github.io/Machine-Learning-cloud-computing_Ucompensar/"


def _es_del_curso(contenido):
    return hashlib.sha256(contenido).hexdigest() == DATOS_SHA256


def _sin_datos(motivo):
    return (f"No pude dejar listos los datos: {motivo}.\n"
            f"Descarga clientes.csv desde la portada del curso ({PORTADA}), súbelo a Colab con el "
            "ícono de carpeta de la izquierda (Subir al almacenamiento de la sesión) y vuelve a "
            "ejecutar esta celda.")


def _leer_url():
    # Solo al preparar el material: la variable apunta a una copia local del mismo archivo
    local = os.environ.get("YARUMO_CSV_LOCAL")
    if local:
        return Path(local).read_bytes()
    with urllib.request.urlopen(DATOS_URL, timeout=30) as respuesta:
        return respuesta.read()


def descargar_datos(carpeta="datos"):
    """Deja <carpeta>/clientes.csv, idéntico al del curso, y devuelve su ruta."""
    ruta = Path(carpeta) / "clientes.csv"
    subido = Path("clientes.csv")  # por si se subió a mano a la carpeta de trabajo de Colab
    if ruta.exists() and _es_del_curso(ruta.read_bytes()):
        contenido, origen = ruta.read_bytes(), "ya estaban"
    elif subido.exists() and _es_del_curso(subido.read_bytes()):
        contenido, origen = subido.read_bytes(), "copiados del archivo que subiste"
    else:
        try:
            contenido = _leer_url()
        except OSError as error:
            raise RuntimeError(_sin_datos(f"no se pudo descargar el archivo ({error})")) from None
        if not _es_del_curso(contenido):
            raise RuntimeError(_sin_datos("el archivo descargado no es el del curso"))
        origen = "descargados del repositorio del curso"
    if origen != "ya estaban":
        ruta.parent.mkdir(parents=True, exist_ok=True)
        temporal = ruta.with_suffix(".tmp")  # sin archivos a medias si algo falla al escribir
        temporal.write_bytes(contenido)
        temporal.replace(ruta)
    lineas = contenido.decode("utf-8").splitlines()
    print(f"Listo: {ruta} con {len(lineas) - 1} clientes y {len(lineas[0].split(','))} columnas, {origen}")
    return ruta


"""Ayudas comunes a todos los notebooks del curso (van en la celda oculta, tras descarga_datos.py).

Las de cada sesión están en ayudas_sesionN.py.
"""


def _aviso_version():
    """Las diapositivas se ejecutaron con pandas 2.2 y scikit-learn 1.6: si Colab trae otras, se dice."""
    import pandas as pd
    import sklearn

    if pd.__version__.split(".")[:2] != ["2", "2"] or sklearn.__version__.split(".")[:2] != ["1", "6"]:
        print(f"Nota: este Colab trae pandas {pd.__version__} y scikit-learn {sklearn.__version__}; "
              "las diapositivas se hicieron con pandas 2.2 y scikit-learn 1.6. Si alguna salida "
              "se ve un poco distinta, es por eso.")


_aviso_version()


"""Sesión 2: estilo de las gráficas del curso y los verificadores de la práctica.

Van dentro de la celda oculta del notebook, junto con descarga_datos.py y ayudas.py. Cada
verificador recibe lo que el estudiante calculó, dice qué está bien y, en lo demás, da una
pista sin la respuesta. Nunca se cae: un «...» sin reemplazar o un tipo inesperado se avisa.

Los valores esperados se calculan aquí, en tiempo de ejecución, desde datos/clientes.csv, con
la misma preparación y la misma semilla de la clase (y se guardan para no repetir el cálculo).
Tolerancias: ejercicio 1, ±0,5 GB en el MAE y ±0,005 en cada R²; ejercicio 2, conteos (con ±2
de tolerancia por si otra versión de scikit-learn cambia un empate); ejercicio 3, el umbral exacto de la
lista y ±2 llamadas; ejercicio 4, la profundidad exacta y ±0,005 en el F1 medio.
"""


def _estilo_graficas():
    """Colores del curso y letras que se leen en Colab y proyectadas."""
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    plt.rcParams.update({
        "axes.prop_cycle": mpl.cycler(color=["#8A2E9E", "#F46201", "#566B7F"]),
        "figure.figsize": (6.4, 3.6),
        "figure.constrained_layout.use": True,
        "font.size": 13,
        "axes.titlesize": 15,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#94A3B8",
        "axes.labelcolor": "#334155",
        "xtick.color": "#475569",
        "ytick.color": "#475569",
        "text.color": "#1E293B",
    })


_estilo_graficas()

_CACHE = {}   # lo que ya se calculó: los datos preparados y los modelos de referencia


def _datos():
    """Las particiones de la clase (regresión y clasificación), o None con aviso."""
    if "datos" in _CACHE:
        return _CACHE["datos"]
    import pandas as pd
    from sklearn.model_selection import train_test_split

    try:
        clientes = pd.read_csv("datos/clientes.csv")
    except FileNotFoundError:
        print("✗  No encuentro datos/clientes.csv: ejecuta primero la celda que descarga los datos "
              "(Entorno de ejecución → Ejecutar anteriores)")
        return None
    X = pd.get_dummies(clientes.drop(columns=["id_cliente", "consumo_gb", "cancelo"]),
                       drop_first=True, dtype=int)
    y = clientes["consumo_gb"]
    reg = train_test_split(X, y, test_size=0.25, random_state=2026)
    Xc = pd.get_dummies(clientes.drop(columns=["id_cliente", "cancelo"]), drop_first=True, dtype=int)
    yc = (clientes["cancelo"] == "Sí").astype(int)
    clf = train_test_split(Xc, yc, test_size=0.25, random_state=2026, stratify=yc)
    _CACHE["datos"] = {"reg": reg, "clf": clf}
    return _CACHE["datos"]


def _sobrantes(sueltos, otros, nombres, ejemplo):
    """Avisa de valores sin nombre o con otro nombre. True si hubo alguno."""
    if sueltos:
        print(f"✗  Pasa los valores por nombre, como en la plantilla: {ejemplo}")
        return True
    if otros:
        print(f"✗  No conozco {', '.join(repr(k) for k in otros)}: usa estos nombres, en "
              f"minúsculas y sin tildes: {', '.join(n + '=' for n in nombres)}")
        return True
    return False


def _seguro(verificador):
    """Red de seguridad: si algo imprevisto falla al revisar, se avisa sin traceback."""
    import functools

    @functools.wraps(verificador)
    def envoltura(*args, **kwargs):
        try:
            return verificador(*args, **kwargs)
        except Exception as error:   # noqa: BLE001 — el estudiante nunca ve un traceback
            print(f"✗  No pude revisar lo que entregaste ({type(error).__name__}). Revisa que cada valor "
                  "sea el resultado de un cálculo (un número) y vuelve a ejecutar la celda")
    return envoltura


def _un_numero(x):
    """(valor, None) si x es un número (o algo con un solo número); si no, (None, motivo)."""
    import math
    import numbers
    import numpy as np
    import pandas as pd

    if x is Ellipsis:
        return None, "todavía tiene los puntos suspensivos (...) de la plantilla: reemplázalos por tu cálculo"
    if isinstance(x, pd.DataFrame):
        if x.size != 1:
            return None, f"es una tabla de {x.shape[0]} × {x.shape[1]}; hace falta un solo número"
        x = x.iloc[0, 0]
    if isinstance(x, (pd.Series, pd.Index, np.ndarray, list, tuple)):
        arreglo = np.asarray(x, dtype=object)
        if arreglo.shape == (2, 2):
            return None, ("es la matriz de confusión entera: saca sus cuatro números con .ravel() "
                          "(en orden VN, FP, FN, VP)")
        valores = list(np.ravel(arreglo))
        if len(valores) == 0:
            return None, "está vacío"
        if len(valores) > 1:
            if len(valores) in (2000, 6000):
                return None, ("es una predicción por cliente; falta resumirla en un número "
                              "(una métrica o un conteo)")
            return None, f"tiene {len(valores)} valores; hace falta uno solo"
        x = valores[0]
        if x is Ellipsis:
            return None, "todavía tiene los puntos suspensivos (...) de la plantilla: reemplázalos por tu cálculo"
    if x is None:
        return None, "está vacío (None): ¿le asignaste el resultado de un print?"
    if callable(x):
        return None, "es una función, no su resultado: ¿faltaron los paréntesis o los datos?"
    if isinstance(x, (bool, np.bool_)):
        return None, "es True o False; hace falta un número"
    if isinstance(x, str):
        return None, "es texto; escríbelo como número, sin comillas y con punto decimal"
    if hasattr(x, "fit") and hasattr(x, "get_params"):
        return None, "es un modelo, no un número: ¿te faltó calcular la métrica?"
    if not isinstance(x, (numbers.Real, np.number)) or isinstance(x, np.complexfloating):
        return None, f"no es un número real (es {type(x).__name__})"
    try:
        valor = float(x)
    except OverflowError:
        return None, "es un número demasiado grande para ser una métrica o un conteo"
    if math.isnan(valor):
        return None, "salió nan: revisa el cálculo"
    if math.isinf(valor):
        return None, "salió infinito: ¿dividiste entre cero?"
    return valor, None


def _cerca(a, b, tolerancia):
    return a is not None and b is not None and abs(a - b) <= tolerancia


def _niveles(p):
    """«1 nivel», «6 niveles»."""
    return f"{p} nivel" if p == 1 else f"{p} niveles"


def _coma(x, decimales=3):
    """Un número con coma decimal, para los mensajes."""
    return f"{x:.{decimales}f}".replace(".", ",").replace("-", "−")


def _leer(entregados, nombres):
    """Convierte cada valor entregado en número; imprime los problemas. Devuelve {nombre: valor}."""
    valores, problemas = {}, {}
    for nombre in nombres:
        x = entregados[nombre]
        if x is Ellipsis:
            print(f"·  {nombre}: todavía tiene los puntos suspensivos (...) de la plantilla: reemplázalos por tu cálculo")
            continue
        valor, problema = _un_numero(x)
        if problema:
            problemas.setdefault(problema, []).append(nombre)
        else:
            valores[nombre] = valor
    for problema, quienes in problemas.items():   # la misma pista una sola vez
        print(f"✗  {', '.join(quienes)}: {problema}")
    return valores


# ---------------------------------------------------------------------------------------------
# Ejercicio 1 · Un árbol con límite (regresión)
# ---------------------------------------------------------------------------------------------

def _referencia_arbol():
    """MAE, RMSE, MSE y R² de los árboles de 1 a 15 niveles y del sin límite, y de la clase."""
    if "arbol" in _CACHE:
        return _CACHE["arbol"]
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.tree import DecisionTreeRegressor

    X_e, X_p, y_e, y_p = _datos()["reg"]
    ref = {}
    for p in list(range(1, 16)) + [None]:
        m = DecisionTreeRegressor(max_depth=p, random_state=2026).fit(X_e, y_e)
        pe, pp = m.predict(X_e), m.predict(X_p)
        ref[p] = {"mae": mean_absolute_error(y_p, pp), "mae_entrena": mean_absolute_error(y_e, pe),
                  "mse": mean_squared_error(y_p, pp), "rmse": mean_squared_error(y_p, pp) ** 0.5,
                  "r2_entrenamiento": r2_score(y_e, pe), "r2_prueba": r2_score(y_p, pp),
                  "r2_al_reves": r2_score(pp, y_p), "r2_entrena_al_reves": r2_score(pe, y_e)}
    _CACHE["arbol"] = ref
    return ref


@_seguro
def revisar_arbol(*sueltos, mae=..., r2_entrenamiento=..., r2_prueba=..., **otros):
    """Ejercicio 1: el MAE en prueba y el R² en entrenamiento y en prueba del árbol de 6 niveles."""
    nombres = ["mae", "r2_entrenamiento", "r2_prueba"]
    if _sobrantes(sueltos, otros, nombres, "revisar_arbol(mae=mae_prueba, r2_entrenamiento=r2_entrena6, "
                                           "r2_prueba=r2_prueba6)"):
        return
    if _datos() is None:
        return
    vals = _leer({"mae": mae, "r2_entrenamiento": r2_entrenamiento, "r2_prueba": r2_prueba}, nombres)
    if not vals:
        return
    ref = _referencia_arbol()
    bien = ref[6]
    tol = {"mae": 0.5, "r2_entrenamiento": 0.005, "r2_prueba": 0.005}
    # ¿Qué otro árbol produce los valores entregados? Se decide con todos juntos: los R² de árboles
    # vecinos se parecen, y un solo valor puede coincidir por casualidad con otra profundidad.
    fina = {"mae": 0.1, "r2_entrenamiento": 0.001, "r2_prueba": 0.001}
    aciertos = {p: sum(_cerca(vals[n], r[n], fina[n]) for n in vals) for p, r in ref.items() if p != 6}
    otra, n_otra = max(aciertos.items(), key=lambda par: par[1])
    if n_otra < min(2, len(vals)) or list(aciertos.values()).count(n_otra) > 1:
        otra = False          # ninguna profundidad (o varias) explica los valores
    bien_ahora = {}
    for nombre in nombres:
        if nombre not in vals:
            continue
        v = vals[nombre]
        # un valor «dentro de la tolerancia» que está más cerca del de otra profundidad no cuenta como bien
        de_otra = otra is not False and abs(v - ref[otra][nombre]) < abs(v - bien[nombre])
        bien_ahora[nombre] = _cerca(v, bien[nombre], tol[nombre]) and not de_otra
        if bien_ahora[nombre]:
            print(f"✓  {nombre}: bien")
            continue
        # ¿De qué otro árbol o de qué otra cuenta es?
        pista = None
        if otra is not False and not _cerca(v, bien[nombre], fina[nombre]) and _cerca(v, ref[otra][nombre], fina[nombre]):
            pista = (f"los valores son los de un árbol {'sin límite' if otra is None else 'de ' + _niveles(otra)}: "
                     "¿pusiste max_depth=6?")
        elif nombre == "mae":
            if _cerca(v, bien["mae_entrena"], 0.5):
                pista = "ese es el MAE del entrenamiento; el que importa es el de la prueba (y_prueba)"
            elif _cerca(v, bien["rmse"], 0.5):
                pista = "esa es la RMSE: el MAE promedia los errores sin elevarlos al cuadrado"
            elif _cerca(v, bien["mse"], 5):
                pista = "ese es el error cuadrático medio (en GB²): el MAE promedia los errores absolutos"
        else:
            otro = "r2_prueba" if nombre == "r2_entrenamiento" else "r2_entrenamiento"
            if _cerca(v, bien[otro], 0.001):
                pista = "está cruzado con el otro R²: uno se mide con el entrenamiento y el otro, con la prueba"
            elif 1 < v <= 100 and _cerca(v / 100, bien[nombre], 0.005):
                pista = "el R² se escribe como proporción (1 es un modelo perfecto), no como porcentaje"
            elif nombre == "r2_prueba" and _cerca(v, bien["r2_al_reves"], 0.005):
                pista = "r2_score recibe primero lo real y después lo predicho: r2_score(y_prueba, pred_prueba)"
            elif nombre == "r2_entrenamiento" and _cerca(v, bien["r2_entrena_al_reves"], 0.005):
                pista = "r2_score recibe primero lo real y después lo predicho: r2_score(y_ent, pred_entrena)"
            elif v < 1 and v == round(v, 1) and round(v, 1) == round(bien[nombre], 1):
                pista = "¿redondeaste? deja al menos dos decimales"
        if pista is None:
            clave = nombre
            posibles = [p for p, r in ref.items() if p != 6 and _cerca(v, r[clave], tol[nombre] / 5 if nombre == "mae" else 0.001)]
            if len(posibles) == 1:
                p = posibles[0]
                pista = ("ese es el valor del árbol sin límite: ¿pusiste max_depth=6?" if p is None
                         else f"ese es el valor de un árbol de {_niveles(p)}: ¿pusiste max_depth=6?")
        print(f"✗  {nombre}: {pista or 'no coincide; revisa con qué datos entrenaste y con cuáles mediste'}")
    if all(bien_ahora.get(n) for n in nombres):
        sin = ref[None]
        print(f"\n¡Listo! El árbol de 6 niveles tiene un R² de {_coma(bien['r2_entrenamiento'], 2)} en entrenamiento y "
              f"de {_coma(bien['r2_prueba'], 2)} en prueba; el sin límite, de {_coma(sin['r2_entrenamiento'], 2)} y de "
              f"{_coma(sin['r2_prueba'], 2)}. Escribe en «Tus respuestas» qué te dice esa diferencia.")


# ---------------------------------------------------------------------------------------------
# Clasificación: los modelos de referencia
# ---------------------------------------------------------------------------------------------

def _modelos_c():
    """El bosque, la logística y XGBoost de la clase, ya entrenados."""
    if "modelos_c" in _CACHE:
        return _CACHE["modelos_c"]
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from xgboost import XGBClassifier

    Xc_e, Xc_p, yc_e, yc_p = _datos()["clf"]
    modelos = {
        "bosque": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=2026, n_jobs=-1),
        "logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=2026),
    }
    for m in modelos.values():
        m.fit(Xc_e, yc_e)
    _CACHE["modelos_c"] = modelos
    return modelos


def _matriz(y_real, pred):
    import numpy as np
    y_real, pred = np.asarray(y_real), np.asarray(pred)
    return {"vn": int(((y_real == 0) & (pred == 0)).sum()), "fp": int(((y_real == 0) & (pred == 1)).sum()),
            "fn": int(((y_real == 1) & (pred == 0)).sum()), "vp": int(((y_real == 1) & (pred == 1)).sum())}


# ---------------------------------------------------------------------------------------------
# Ejercicio 2 · La matriz de confusión del bosque
# ---------------------------------------------------------------------------------------------

@_seguro
def revisar_matriz(*sueltos, vp=..., fp=..., fn=..., vn=..., **otros):
    """Ejercicio 2: los cuatro números de la matriz de confusión del bosque, con el umbral de 0,5."""
    nombres = ["vp", "fp", "fn", "vn"]
    if _sobrantes(sueltos, otros, nombres, "revisar_matriz(vp=vp_b, fp=fp_b, fn=fn_b, vn=vn_b)"):
        return
    if _datos() is None:
        return
    entregados = {"vp": vp, "fp": fp, "fn": fn, "vn": vn}
    # si la tupla de .ravel() quedó entera en una sola variable
    import numpy as np
    for nombre, x in entregados.items():
        if isinstance(x, (np.ndarray, list, tuple)) and np.size(x) == 4:
            print(f"✗  {nombre}: tiene los cuatro números juntos; reparte .ravel() en cuatro variables, "
                  "en el orden VN, FP, FN, VP: vn_b, fp_b, fn_b, vp_b = matriz.ravel()")
            return
    vals = _leer(entregados, nombres)
    if not vals:
        return
    no_enteros = [k for k, v in vals.items() if v != int(v) or v < 0]
    if no_enteros:
        print(f"✗  {', '.join(no_enteros)}: cada casilla de la matriz es un conteo de clientes (un entero "
              "de 0 en adelante), no una proporción")
        return
    Xc_e, Xc_p, yc_e, yc_p = _datos()["clf"]
    modelos = _modelos_c()
    bien = _matriz(yc_p, modelos["bosque"].predict(Xc_p))
    tol = 2
    if all(_cerca(vals.get(k), bien[k], tol) for k in nombres):
        print("✓  vp, fp, fn y vn: bien")
        print(f"\n¡Listo! El bosque encontró a {bien['vp']} de los {bien['vp'] + bien['fn']} que cancelaron "
              f"y señaló a {bien['fp']} que no cancelaron. Escríbelo en «Tus respuestas».")
        return
    completos = len(vals) == 4
    if completos:
        total = sum(vals.values())
        if total == len(yc_e):
            print(f"✗  Los cuatro suman {int(total)}: son los clientes del entrenamiento. La matriz se calcula "
                  "con la prueba (yc_prueba y la predicción de Xc_prueba)")
            return
        if total != len(yc_p) and sum(_cerca(vals[k], bien[k], tol) for k in nombres) < 3:
            print(f"✗  Los cuatro suman {int(total)}, y la prueba tiene {len(yc_p)} clientes: revisa que la "
                  "matriz sea de yc_prueba contra la predicción de Xc_prueba")
            return
        # ¿Es la matriz de otro modelo?
        for nombre, m in modelos.items():
            if nombre != "bosque" and all(_cerca(vals[k], v, tol) for k, v in _matriz(yc_p, m.predict(Xc_p)).items()):
                print(f"✗  Esos son los números de {'la logística' if nombre == 'logística' else nombre}, no los del bosque: "
                      "predice con bosque_c")
                return
        if vals["vp"] == 0 and vals["fp"] == 0:
            print("✗  Con VP y FP en 0, el modelo dijo «No» a todos, como la línea base: predice con bosque_c")
            return
        # ¿Están los números bien, pero en otro lugar?
        if sorted(vals.values()) == sorted(bien.values()):
            if (_cerca(vals["fp"], bien["fn"], 0) and _cerca(vals["fn"], bien["fp"], 0)
                    and _cerca(vals["vp"], bien["vp"], 0) and _cerca(vals["vn"], bien["vn"], 0)):
                print("✗  fp y fn están cruzados: confusion_matrix recibe primero lo real y después lo predicho "
                      "(yc_prueba, pred_bosque)")
            else:
                print("✗  Están los cuatro números, pero no en su lugar: el orden de .ravel() es VN, FP, FN, VP")
            return
    for k in nombres:
        if k in vals:
            print(f"{'✓' if _cerca(vals[k], bien[k], tol) else '✗'}  {k}: "
                  f"{'bien' if _cerca(vals[k], bien[k], tol) else 'no coincide'}")
    malos = [k for k, v in vals.items() if not _cerca(v, bien[k], tol)]
    if not malos:
        return      # lo entregado está bien; solo falta lo que no era un número: ya se avisó arriba
    if len(vals) == 4 and len(malos) == 1:
        print(f"·  Los otros tres coinciden: revisa cómo sacaste {malos[0]}")
    elif "vp" in vals and "fn" in vals and _cerca(vals["vp"] + vals["fn"], bien["vp"] + bien["fn"], 0):
        print("·  vp + fn sí suman los que cancelaron: revisa con qué modelo y con qué umbral predijiste")
    else:
        print("·  Pista: predice con bosque_c.predict(Xc_prueba) y calcula confusion_matrix(yc_prueba, …)")


# ---------------------------------------------------------------------------------------------
# Ejercicio 3 · El umbral de la gerente
# ---------------------------------------------------------------------------------------------

_UMBRALES = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20, 0.15]


def _tabla_umbral(nombre_modelo):
    """Sensibilidad, llamadas y encontrados de cada umbral de la lista, con un modelo."""
    clave = f"umbral_{nombre_modelo}"
    if clave in _CACHE:
        return _CACHE[clave]
    import numpy as np
    Xc_e, Xc_p, yc_e, yc_p = _datos()["clf"]
    prob = _modelos_c()[nombre_modelo].predict_proba(Xc_p)[:, 1]
    y = np.asarray(yc_p)
    tabla = {}
    for u in _UMBRALES:
        pred = (prob >= u).astype(int)
        tabla[u] = {"sensibilidad": float(pred[y == 1].mean()), "llamadas": int(pred.sum()),
                    "encontrados": int(pred[y == 1].sum())}
    _CACHE[clave] = tabla
    return tabla


@_seguro
def revisar_umbral(*sueltos, umbral=..., llamadas=..., **otros):
    """Ejercicio 3: el umbral más alto con el que XGBoost encuentra al menos a 7 de cada 10, y sus llamadas."""
    nombres = ["umbral", "llamadas"]
    if _sobrantes(sueltos, otros, nombres, "revisar_umbral(umbral=umbral_elegido, llamadas=llamadas_elegido)"):
        return
    if _datos() is None:
        return
    vals = _leer({"umbral": umbral, "llamadas": llamadas}, nombres)
    if not vals:
        return
    tabla = _tabla_umbral("XGBoost")
    cumplen = [u for u in _UMBRALES if tabla[u]["sensibilidad"] >= 0.70]
    mejor = max(cumplen)
    u = vals.get("umbral")
    elegido = None
    if u is not None:
        if 1 < u <= 100 and any(_cerca(u / 100, x, 1e-9) for x in _UMBRALES):
            print("✗  umbral: es una probabilidad entre 0 y 1, no un porcentaje")
        elif not any(_cerca(u, x, 1e-9) for x in _UMBRALES):
            print("✗  umbral: elige uno de los umbrales del ciclo (0.50, 0.45, …, 0.15)")
        else:
            elegido = min(_UMBRALES, key=lambda x: abs(x - u))
            s = tabla[elegido]["sensibilidad"]
            if elegido == mejor:
                print("✓  umbral: bien")
            elif s < 0.70:
                print(f"✗  umbral: con {_coma(elegido, 2)}, XGBoost encuentra al {_coma(s * 100, 1)} % de los que cancelan: "
                      "no llega a la meta del 70 %. Baja el umbral")
            else:
                print(f"✗  umbral: con {_coma(elegido, 2)} sí llega a la meta, pero hay un umbral más alto que "
                      "también llega, con menos llamadas")
    n = vals.get("llamadas")
    if n is not None:
        if 0 < n < 1:
            print("✗  llamadas: es un conteo de clientes (cuántos 1 hay en pred), no una proporción")
        elif elegido is None:
            print("·  llamadas: la reviso cuando el umbral sea uno de la lista")
        else:
            fila = tabla[elegido]
            if _cerca(n, fila["llamadas"], 2):
                print(f"✓  llamadas: bien para el umbral de {_coma(elegido, 2)}")
            elif _cerca(n, fila["encontrados"], 2):
                print("✗  llamadas: esos son los encontrados (los que cancelaron); las llamadas son todos los "
                      "clientes con 1 en pred: pred.sum()")
            elif _cerca(n, 2000 - fila["llamadas"], 2):
                print("✗  llamadas: contaste los 0; las llamadas son los 1 de pred")
            elif _cerca(n, _tabla_umbral("logística")[elegido]["llamadas"], 2):
                print("✗  llamadas: son las de la logística; aquí van las de XGBoost (prob_xgb)")
            else:
                otro = [x for x in _UMBRALES if _cerca(n, tabla[x]["llamadas"], 2)]
                print("✗  llamadas: " + (f"son las del umbral de {_coma(otro[0], 2)}, no las del que elegiste"
                                         if otro else "no coinciden con las de ese umbral: cuenta los 1 de pred"))
    if elegido == mejor and n is not None and _cerca(n, tabla[mejor]["llamadas"], 2):
        f = tabla[mejor]
        print(f"\n¡Listo! Con {_coma(mejor, 2)}, XGBoost encuentra a {f['encontrados']} de los que cancelaron "
              f"({_coma(f['sensibilidad'] * 100, 1)} %) con {f['llamadas']} llamadas. Compara su exactitud con la de 0,5 "
              "en «Tus respuestas».")


# ---------------------------------------------------------------------------------------------
# Ejercicio 4 · La profundidad que elige la validación cruzada
# ---------------------------------------------------------------------------------------------

def _referencia_profundidad():
    if "profundidad" in _CACHE:
        return _CACHE["profundidad"]
    from sklearn.metrics import f1_score
    from sklearn.model_selection import StratifiedKFold, cross_validate
    from sklearn.tree import DecisionTreeClassifier

    Xc_e, Xc_p, yc_e, yc_p = _datos()["clf"]
    pliegues = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
    ref = {}
    for p in range(1, 11):
        m = DecisionTreeClassifier(max_depth=p, random_state=2026)
        cv = cross_validate(m, Xc_e, yc_e, cv=pliegues, scoring=["f1", "accuracy", "recall"],
                            return_train_score=True)
        m.fit(Xc_e, yc_e)
        cv5 = cross_validate(DecisionTreeClassifier(max_depth=p, random_state=2026), Xc_e, yc_e, cv=5, scoring="f1")
        ref[p] = {"f1": cv["test_f1"].mean(), "f1_cv5": cv5["test_score"].mean(), "exactitud": cv["test_accuracy"].mean(),
                  "sensibilidad": cv["test_recall"].mean(), "f1_entrena": cv["train_f1"].mean(),
                  "f1_prueba": f1_score(yc_p, m.predict(Xc_p))}
    _CACHE["profundidad"] = ref
    return ref


@_seguro
def revisar_profundidad(*sueltos, profundidad=..., f1_medio=..., **otros):
    """Ejercicio 4: la profundidad con el F1 medio más alto en la validación cruzada, y ese F1."""
    nombres = ["profundidad", "f1_medio"]
    if _sobrantes(sueltos, otros, nombres, "revisar_profundidad(profundidad=mejor_profundidad, "
                                           "f1_medio=mejor_f1)"):
        return
    if _datos() is None:
        return
    vals = _leer({"profundidad": profundidad, "f1_medio": f1_medio}, nombres)
    if not vals:
        return
    ref = _referencia_profundidad()
    mejor = max(ref, key=lambda p: ref[p]["f1"])
    p = vals.get("profundidad")
    f = vals.get("f1_medio")
    mejor_cv5 = max(ref, key=lambda q: ref[q]["f1_cv5"])
    if p is not None and f is not None and p == mejor_cv5 and mejor_cv5 != mejor and \
            _cerca(f, ref[mejor_cv5]["f1_cv5"], 0.001):
        print("✗  Esos F1 salen de cv=5, que reparte los pliegues de otra manera: usa cv=pliegues (los pliegues de "
              "la clase) para que todos obtengamos lo mismo")
        return
    if p is not None:
        if p != int(p) or not 1 <= p <= 10:
            print("✗  profundidad: es un número entero de niveles, de 1 a 10")
            p = None
        else:
            p = int(p)
            if p == mejor:
                print("✓  profundidad: bien")
            elif p == max(ref, key=lambda q: ref[q]["f1_entrena"]):
                print("✗  profundidad: esa es la que mejor le va en el entrenamiento, no en la validación: "
                      "mide con cross_val_score")
            elif p == max(ref, key=lambda q: ref[q]["f1_prueba"]):
                print("✗  profundidad: esa es la que mejor le va en la prueba; aquí se elige con la validación "
                      "cruzada, sobre el entrenamiento")
            elif abs(ref[p]["f1"] - ref[mejor]["f1"]) < 0.01:
                print(f"✗  profundidad: casi: con {_niveles(p)}, el F1 medio es {_coma(ref[p]['f1'])}; hay una "
                      "profundidad un poco mejor")
            else:
                print(f"✗  profundidad: con {_niveles(p)}, el F1 medio es {_coma(ref[p]['f1'])}; busca el más alto")
    if f is not None and p is None:
        print("·  f1_medio: lo reviso cuando la profundidad sea un entero de 1 a 10")
    elif f is not None:
        objetivo = ref[p]["f1"]
        if _cerca(f, objetivo, 0.005):
            print("✓  f1_medio: bien" + ("" if p == mejor else f" para {_niveles(p)}"))
        elif 1 < f <= 100 and _cerca(f / 100, objetivo, 0.005):
            print("✗  f1_medio: el F1 se escribe como proporción, entre 0 y 1, no como porcentaje")
        elif _cerca(f, ref[p]["exactitud"], 0.005):
            print("✗  f1_medio: esa es la exactitud; pide scoring=\"f1\" en cross_val_score")
        elif _cerca(f, ref[p]["sensibilidad"], 0.005):
            print("✗  f1_medio: esa es la sensibilidad; pide scoring=\"f1\"")
        elif _cerca(f, ref[p]["f1_prueba"], 0.001):
            print("✗  f1_medio: ese es el F1 en la prueba; aquí va el promedio de los 5 pliegues")
        else:
            print("✗  f1_medio: no coincide: es el promedio (.mean()) de los 5 F1 de cross_val_score, con "
                  "cv=pliegues y scoring=\"f1\"")
    if p == mejor and f is not None and _cerca(f, ref[mejor]["f1"], 0.005):
        print(f"\n¡Listo! La validación cruzada elige {mejor} niveles. Explica en «Tus respuestas» por qué no se "
              "elige con la prueba.")

# Descarga datos/clientes.csv: los mismos 8000 clientes de la sesión 1
ruta = descargar_datos("datos")

## 1 · Los datos

Cargamos los clientes como en la sesión 1. Hoy usamos también **scikit-learn**, la biblioteca de
aprendizaje automático que Colab ya trae instalada, y **XGBoost**, que también viene en Colab.

**Repaso: lo nuevo de pandas y numpy que aparece hoy.** No hace falta memorizarlo; vuelve aquí cuando lo veas.

| Escribes | Qué hace |
|---|---|
| `tabla.drop(columns=["a", "b"])` | La tabla sin esas columnas (no cambia la original) |
| `tabla.loc[filas, columnas]` | Elige filas y columnas por su nombre, por ejemplo `clientes.loc[[0, 1], "contrato"]` |
| `tabla.set_index("id_cliente")` | Usa esa columna como nombre de las filas |
| `tabla.assign(nueva=valores)` | Una copia de la tabla con una columna más |
| `tabla.T` | La tabla traspuesta: las filas pasan a columnas |
| `tabla.round(3)` | Redondea a 3 decimales (con un diccionario, cada columna a los suyos) |
| `np.sqrt(x)` | La raíz cuadrada, con numpy (`np`) |
| `f"{x:.2%}"` | Escribe `x` como porcentaje, con 2 decimales: 0.7645 → 76.45% |

In [ ]:
# [cargar]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost

clientes = pd.read_csv("datos/clientes.csv")
print(f"scikit-learn {sklearn.__version__} · xgboost {xgboost.__version__}")
clientes.shape

### Diccionario de datos (el mismo de la sesión 1)

| Columna | Qué registra | Valores o unidad |
|---|---|---|
| <code>id&#95;cliente</code> | Seudónimo del cliente | C00001… |
| `ciudad` | Ciudad del servicio | Bogotá, Medellín, Cali, Barranquilla, Pereira |
| `estrato` | Estrato de la **dirección del servicio** (lo asigna el municipio a la vivienda; no es el ingreso) | 1 a 6 |
| `edad` | Edad del titular | años |
| <code>tipo&#95;cliente</code> | Hogar o empresa pequeña | Hogar, Pyme |
| `usuarios` | Personas que usan la conexión (declarado al contratar) | conteo |
| `tecnologia` | Tecnología de acceso | Fibra, Inalámbrica |
| <code>plan&#95;mbps</code> | Velocidad contratada | 100, 300, 500, 900 Mbps |
| `contrato` | Permanencia pactada | Mes a mes, 12 meses, 24 meses |
| <code>antiguedad&#95;meses</code> | Meses como cliente | 1 a 96 |
| `tv`, `movil` | Si tiene también TV o línea móvil con Yarumo | Sí / No |
| <code>medio&#95;pago</code> | Cómo paga | Débito automático, Tarjeta de crédito, Pago en línea, Efectivo |
| `descuento` | Si tiene un descuento vigente (el de bienvenida dura 12 meses; algunos clientes antiguos tienen otras promociones) | Sí / No |
| <code>factura&#95;mensual</code> | Valor mensual facturado | pesos (COP), ficticios |
| <code>consumo&#95;gb</code> | Datos consumidos al mes, en promedio | GB |
| <code>velocidad&#95;pct</code> | Velocidad medida como % de la contratada (promedio de 3 meses) | % |
| <code>caidas&#95;3m</code> | Caídas del servicio en los últimos 3 meses | conteo |
| <code>llamadas&#95;soporte&#95;3m</code> | Llamadas a soporte en los últimos 3 meses | conteo |
| <code>pagos&#95;tardios&#95;12m</code> | Facturas pagadas tarde en los últimos 12 meses | conteo |
| `cancelo` | **Etiqueta**: si canceló entre el 1 de julio y el 30 de septiembre de 2026 | Sí / No |

## Las preguntas de hoy

Hoy entrenamos modelos **supervisados**: aprenden de ejemplos que traen la respuesta, la **etiqueta**.

- **Regresión**, cuando la etiqueta es un número. El jefe de TI planea la capacidad de la red y pregunta:
  *¿cuántos GB al mes consumirá cada cliente?* La etiqueta es `consumo_gb`.
- **Clasificación**, cuando la etiqueta es una categoría. La gerente de experiencia del cliente pregunta,
  como en la sesión 1: *¿quién va a cancelar?* La etiqueta es `cancelo`.
- **Nube**, cuando la pregunta es de acceso. El oficial de protección de datos pregunta: *¿quién puede ver
  qué en Azure?* No se responde con un modelo sino con roles y directivas (parte 3).

Cada modelo se mide en clientes que **no vio** al entrenar y se compara con su **línea base**.

---
# Parte 1 · Regresión: ¿cuántos GB consumirá cada cliente?

## 2 · Las características, en números

Las **características** (las columnas que el modelo puede mirar; aquí las llamamos **predictoras**) tienen que
ser números, porque los modelos de scikit-learn solo leen números. Una columna de texto, como `contrato`, se vuelve varias
columnas de 0 y 1, una por categoría: es la codificación ***one-hot***. `pd.get_dummies` la hace; con
`drop_first=True` omite la primera categoría de cada columna (en orden alfabético), porque se deduce
de las demás: si un cliente no es «24 meses» ni «Mes a mes», es «12 meses».

In [ ]:
# [predictoras]
# Todas las columnas menos el seudónimo, la etiqueta de hoy (consumo_gb)
# y cancelo, que se supo después (de julio a septiembre)
predictoras = clientes.drop(
    columns=["id_cliente", "consumo_gb", "cancelo"])
X = pd.get_dummies(predictoras, drop_first=True, dtype=int)   # one-hot
y = clientes["consumo_gb"]                 # la etiqueta: GB al mes
print(f"{predictoras.shape[1]} columnas → {X.shape[1]} columnas numéricas")

In [ ]:
# [onehot]
# Tres clientes, antes y después: cada categoría es una columna de 0 y 1
print(clientes.loc[[0, 1, 2], "contrato"].to_list())
X.loc[[0, 1, 2], ["contrato_24 meses", "contrato_Mes a mes"]]

El primer cliente tiene contrato a 12 meses: las dos columnas valen 0. El segundo es «Mes a mes» y el
tercero, «24 meses». La ciudad, el medio de pago y las columnas Sí/No se codifican igual.

Separamos los clientes como en la sesión 1: la cuarta parte se guarda para la **prueba**, con la misma
semilla (2026), y el modelo nunca la ve al entrenar. Esta vez sin `stratify`, porque la etiqueta es un
número: no son exactamente los mismos 2000 clientes de prueba de la sesión 1.

In [ ]:
# [particion]
from sklearn.model_selection import train_test_split
X_ent, X_prueba, y_ent, y_prueba = train_test_split(
    X, y, test_size=0.25, random_state=2026)
print(f"Entrenamiento: {len(X_ent)} clientes · prueba: {len(X_prueba)}")

## 3 · La línea base y las tres métricas

La **línea base** de una regresión predice el promedio del entrenamiento para todos los clientes, sin
mirar ninguna columna (`DummyRegressor`). Un modelo que no le gane no sirve. Tres métricas, siempre
en el conjunto de **prueba**:

- **MAE** (error absoluto medio): cuántos GB se equivoca, en promedio, por cliente:
  $\text{MAE} = \frac{1}{n}\sum |y_i - \hat y_i|$.
- **RMSE** (raíz del error cuadrático medio): también en GB, pero eleva al cuadrado antes de promediar,
  así que castiga más los errores grandes: $\text{RMSE} = \sqrt{\frac{1}{n}\sum (y_i - \hat y_i)^2}$.
- **R²**: la parte de la variación del consumo que el modelo reproduce. La línea base da 0; un modelo
  perfecto, 1. No es un porcentaje de aciertos.

In [ ]:
# [linea_base_reg]
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (mean_absolute_error, r2_score,
                             root_mean_squared_error)
base = DummyRegressor()                    # predice el promedio, siempre
base.fit(X_ent, y_ent)                     # solo aprende el promedio
pred_base = base.predict(X_prueba)
print(f"Predice {pred_base[0]:.1f} GB para todos")
print(f"MAE  = {mean_absolute_error(y_prueba, pred_base):.1f} GB")
print(f"RMSE = {root_mean_squared_error(y_prueba, pred_base):.1f} GB")
print(f"R²   = {r2_score(y_prueba, pred_base):.3f}")

Para no repetir esas líneas con cada modelo, una función las junta: entrena con el conjunto de
entrenamiento, predice el de prueba y devuelve las tres métricas.

In [ ]:
# [medir]
def medir(modelo):
    """Entrena el modelo, predice la prueba y devuelve MAE, RMSE y R²."""
    modelo.fit(X_ent, y_ent)               # aprende de los ejemplos
    pred = modelo.predict(X_prueba)        # predice clientes que no vio
    return {"MAE": mean_absolute_error(y_prueba, pred),
            "RMSE": root_mean_squared_error(y_prueba, pred),
            "R²": r2_score(y_prueba, pred)}

resultados = {"Línea base": medir(DummyRegressor())}

## 4 · Regresión lineal

La regresión lineal suma cada columna multiplicada por un coeficiente: una recta (un plano, con
muchas columnas) que deja la menor suma de errores al cuadrado.

In [ ]:
# [lineal]
from sklearn.linear_model import LinearRegression
lineal = LinearRegression()
resultados["Lineal"] = medir(lineal)
pd.DataFrame(resultados).T.round(3)

## 5 · K vecinos más cercanos (KNN) y por qué hay que escalar

KNN predice el consumo de un cliente con el promedio de sus *k* vecinos más parecidos del
entrenamiento. «Parecido» es una **distancia**, y la distancia suma diferencias de columnas con
unidades distintas: años, personas y pesos. Mira tres clientes reales.

In [ ]:
# [tres]
tres = clientes.set_index("id_cliente").loc[
    ["C00053", "C00611", "C01689"],
    ["edad", "usuarios", "factura_mensual", "consumo_gb"]]
tres                                   # A, B y C, en ese orden

A y B tienen la misma edad y los mismos usuarios, y consumen parecido; B paga 20 000 pesos más porque
tiene un plan más rápido. C paga lo mismo que A, pero tiene 38 años más y consume mucho menos. ¿Cuál
es el vecino más cercano de A?

In [ ]:
# [escala]
columnas = ["edad", "usuarios", "factura_mensual"]
def distancia(datos, a, b):                # la distancia en línea recta
    return np.sqrt(((datos.loc[a] - datos.loc[b]) ** 2).sum())
crudos = tres[columnas]
print(f"Sin escalar: A–B {distancia(crudos, 'C00053', 'C00611'):.2f}",
      f"· A–C {distancia(crudos, 'C00053', 'C01689'):.2f}")
escalados = crudos / X_ent[columnas].std()   # en desviaciones estándar
print(f"Escalados:   A–B {distancia(escalados, 'C00053', 'C00611'):.2f}",
      f"· A–C {distancia(escalados, 'C00053', 'C01689'):.2f}")

Con solo estas tres columnas y sin escalar, 20 000 pesos de diferencia pesan más que 38 años: para la
distancia, el cliente C de 66 años es el «vecino» de A, aunque consume la octava parte. **Escalar** pone
cada columna en desviaciones estándar: resta su media y divide entre su desviación, las dos aprendidas
**solo** del entrenamiento (`StandardScaler`). Así, 20 000 pesos son media desviación y 38 años, más de
dos y media. (En la celda solo dividimos: restar la media no cambia las distancias.)

Un `Pipeline` (aquí con `make_pipeline`) encadena los pasos: primero escala, luego aplica el modelo.
Al entrenar, el escalador aprende la media y la desviación del entrenamiento; al predecir, usa esas
mismas con la prueba.

In [ ]:
# [knn]
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
knn = make_pipeline(StandardScaler(),      # primero escala, luego vecinos
                    KNeighborsRegressor(n_neighbors=10))
resultados["KNN (10 vecinos)"] = medir(knn)
pd.DataFrame(resultados).T.round(3)

KNN no le gana a la regresión lineal. Con 24 columnas escaladas, todas pesan igual en la distancia,
también las que poco tienen que ver con el consumo (las caídas, el medio de pago, los pagos tardíos),
y esas estorban. Los árboles, en cambio, eligen qué columnas usar.

Antes de ejecutar la celda siguiente, predice: **sin escalar, ¿KNN saldrá mejor o peor?**

In [ ]:
# [knn_sin]
knn_sin = KNeighborsRegressor(n_neighbors=10)   # sin escalar
r_sin = medir(knn_sin)
print(f"KNN sin escalar: MAE {r_sin['MAE']:.1f} GB · R² {r_sin['R²']:.3f}")

En estos datos, sin escalar sale un poco mejor. No es porque escalar esté mal: sin escalar, la distancia
la deciden casi solo la factura y el plan, que resumen bien el consumo (sobre todo el de las empresas
pequeñas), y las columnas que estorban quedan fuera por sus números pequeños. Es suerte de las unidades,
no una regla: **se escala para que el resultado no dependa de las unidades**, no para ganar puntos. En la
clasificación veremos un caso en que no escalar es un desastre.

## 6 · Árbol de decisión, bosque aleatorio y XGBoost

Un **árbol de decisión** parte a los clientes con preguntas sobre una columna («¿tiene más de 3
usuarios?») hasta formar grupos de consumo parecido, y predice el promedio de cada grupo. Sin límite
de profundidad, sigue partiendo hasta que cada hoja tiene casi un solo cliente.

In [ ]:
# [arbol]
from sklearn.tree import DecisionTreeRegressor
arbol = DecisionTreeRegressor(random_state=2026)   # sin límite
resultados["Árbol sin límite"] = medir(arbol)
r2_entrena = r2_score(y_ent, arbol.predict(X_ent))
print(f"R² en entrenamiento: {r2_entrena:.3f}")
print(f"R² en prueba:        {resultados['Árbol sin límite']['R²']:.3f}")

En entrenamiento, el árbol sin límite es perfecto; en prueba, peor que la regresión lineal. Memorizó
a los clientes que vio, con sus rarezas al azar (el ruido): es **sobreajuste** (*overfitting*). Aprender no
es memorizar.

Un **bosque aleatorio** (*random forest*) entrena muchos árboles, cada uno con una muestra al azar (con
repetición) de los clientes, y promedia sus predicciones: los errores de un árbol se compensan con los
de otros. (El bosque de clasificación de la parte 2, además, sortea las columnas en cada pregunta.)
`min_samples_leaf=5` pide al menos 5 clientes por hoja. Los ajustes que se eligen antes de entrenar, como
el número de árboles o su profundidad, se llaman **hiperparámetros** (*hyperparameters*).

In [ ]:
# [bosque]
from sklearn.ensemble import RandomForestRegressor
bosque = RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                               random_state=2026, n_jobs=-1)
resultados["Bosque aleatorio"] = medir(bosque)

**XGBoost** es un método de potenciación del gradiente (*gradient boosting*): entrena árboles
pequeños **en secuencia**, y cada uno corrige lo que los anteriores todavía no aciertan. El
`learning_rate` dice cuánto corrige cada árbol; `max_depth`, cuántas preguntas hace cada uno.

In [ ]:
# [xgboost]
from xgboost import XGBRegressor
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                   random_state=2026)
resultados["XGBoost"] = medir(xgb)
tabla_reg = pd.DataFrame(resultados).T
tabla_reg.round(3)

In [ ]:
# [comparacion_reg]
fig, ax = plt.subplots(figsize=(8.4, 3.9))
orden = tabla_reg.sort_values("MAE", ascending=False)
colores = ["#566B7F" if m == "Línea base" else "#8A2E9E"
           for m in orden.index]
ax.barh(orden.index, orden["MAE"], color=colores)
for i, valor in enumerate(orden["MAE"]):
    ax.text(valor + 2, i, f"{valor:.0f} GB", va="center", fontsize=12)
ax.set_xlim(0, orden["MAE"].max() * 1.18)
ax.set_title("El bosque y XGBoost se equivocan menos")
ax.set_xlabel("Error absoluto medio en prueba (GB por cliente)")
plt.show()

¿Por qué los bosques y XGBoost le ganan a la recta? La gráfica compara, por grupos de edad de los
clientes de **prueba**, el consumo real promedio con lo que predicen la regresión lineal y XGBoost.

In [ ]:
# [curva_edad]
grupos = pd.cut(X_prueba["edad"], bins=[17, 24, 29, 34, 39, 44, 49, 54,
                                        59, 64, 69, 90])
curva = pd.DataFrame({"real": y_prueba,
                      "lineal": lineal.predict(X_prueba),
                      "xgboost": xgb.predict(X_prueba)}).groupby(
    grupos, observed=True).mean()
centros = [21, 27, 32, 37, 42, 47, 52, 57, 62, 67, 75]
fig, ax = plt.subplots(figsize=(8.4, 3.9))
ax.plot(centros, curva["real"], "o", color="#1E293B", label="Real")
ax.plot(centros, curva["lineal"], color="#566B7F", linewidth=2.5,
        label="Regresión lineal")
ax.plot(centros, curva["xgboost"], color="#F46201", linewidth=2.5,
        label="XGBoost")
ax.set_title("La recta no sigue al consumo por edad; XGBoost sí")
ax.set_xlabel("Edad del titular (años)")
ax.set_ylabel("Consumo promedio (GB al mes)")
ax.legend(frameon=False)
plt.show()

La recta se aleja en los extremos: para los de 70 años o más llega a predecir un consumo negativo. ¿Qué
le falta? Probemos dos arreglos: agregarle la **edad al cuadrado** (para que pueda curvarse) o la **edad
multiplicada por los usuarios** (para que el efecto de la edad dependa de cuántas personas usan la
conexión). Antes de ejecutar, predice cuál ayuda más.

In [ ]:
# [interaccion]
intentos = {"edad al cuadrado": X["edad"] ** 2,
            "edad × usuarios": X["edad"] * X["usuarios"]}
for nombre, columna in intentos.items():
    X2 = X.assign(nueva=columna)               # X con una columna más
    recta2 = LinearRegression().fit(X2.loc[X_ent.index], y_ent)
    pred2 = recta2.predict(X2.loc[X_prueba.index])
    mae2 = mean_absolute_error(y_prueba, pred2)
    r2 = r2_score(y_prueba, pred2)
    print(f"Recta + {nombre}: MAE {mae2:.1f} GB · R² {r2:.3f}")

La edad al cuadrado casi no ayuda; la edad × usuarios acerca la recta a los árboles. El consumo no es una
suma de efectos: cumplir años le quita más GB a un hogar de muchas personas que a uno de una o dos. Esa
combinación de columnas es una **interacción**. La regresión lineal solo suma; los árboles combinan
columnas solos, porque cada pregunta se hace dentro del grupo que dejó la anterior.

---
## Ejercicio 1 · Un árbol con límite

El árbol sin límite memorizó: R² de 1 en entrenamiento y mucho menos en prueba. Entrena un árbol de
regresión con **profundidad máxima 6** (`max_depth=6`) y la misma semilla, y calcula su **MAE en
prueba** y su **R² en entrenamiento y en prueba**. Pista: las celdas [linea_base_reg] y [arbol].

Cambia cada `...` (mientras quede uno, la celda da un error que menciona `Ellipsis`). El verificador te
dice qué está bien y, en lo demás, te da una pista. Después responde abajo, en «Tus respuestas»: ¿se
sobreajusta menos que el árbol sin límite? ¿Cómo lo sabes? Si terminas antes, prueba con 4 y con 10
niveles y mira qué pasa con la diferencia entre el entrenamiento y la prueba.

In [ ]:
# [ej1]
arbol6 = DecisionTreeRegressor(max_depth=..., random_state=2026)
arbol6.fit(..., ...)                       # entrena con el entrenamiento
pred_entrena = arbol6.predict(X_ent)
pred_prueba = arbol6.predict(...)          # predice la prueba
mae_prueba = ...                           # el MAE en prueba
r2_entrena6 = ...                          # el R² en entrenamiento
r2_prueba6 = ...                           # el R² en prueba
revisar_arbol(mae=mae_prueba, r2_entrenamiento=r2_entrena6,
              r2_prueba=r2_prueba6)

---
# Parte 2 · Clasificación: ¿quién va a cancelar?

## 7 · Las características y la línea base

Ahora la etiqueta es `cancelo`, que convertimos en 1 (canceló) o 0 (no canceló). Las predictoras (las
características) son todas las demás columnas, ahora **con** `consumo_gb`. Para no mezclarlas con las de la regresión, sus
nombres llevan una **c** (de cancelación): `Xc_ent`, `yc_prueba`… Con `stratify=yc`, el
entrenamiento y la prueba quedan con la misma tasa de cancelación, como en la sesión 1.

In [ ]:
# [preparar_c]
predictoras_c = clientes.drop(columns=["id_cliente", "cancelo"])
Xc = pd.get_dummies(predictoras_c, drop_first=True, dtype=int)
yc = (clientes["cancelo"] == "Sí").astype(int)   # 1 = canceló
Xc_ent, Xc_prueba, yc_ent, yc_prueba = train_test_split(
    Xc, yc, test_size=0.25, random_state=2026, stratify=yc)
print(f"Prueba: {len(yc_prueba)} clientes; cancelaron {yc_prueba.sum()}")

In [ ]:
# [linea_base_c]
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
base_c = DummyClassifier(strategy="most_frequent")   # siempre «No»
base_c.fit(Xc_ent, yc_ent)
pred_base_c = base_c.predict(Xc_prueba)
encontrados_base = ((pred_base_c == 1) & (yc_prueba == 1)).sum()
print(f"Exactitud: {accuracy_score(yc_prueba, pred_base_c):.2%}")
print(f"Encontró a {encontrados_base} de los {yc_prueba.sum()}",
      "que cancelaron")

Es la trampa de la exactitud de la sesión 1: decir siempre «No» acierta tres de cada cuatro veces y no
encuentra a nadie.

## 8 · Regresión logística y la matriz de confusión

La **regresión logística**, pese a su nombre, clasifica: combina las columnas como una regresión lineal
y pasa el resultado por una curva en S que lo deja entre 0 y 1, la **probabilidad** de cancelar. Con
`predict`, decide «canceló» si esa probabilidad es de 0,5 o más. Escalamos primero, como con KNN.

In [ ]:
# [logistica]
from sklearn.linear_model import LogisticRegression
logistica = make_pipeline(StandardScaler(),
                          LogisticRegression(max_iter=1000))
logistica.fit(Xc_ent, yc_ent)
pred_log = logistica.predict(Xc_prueba)    # 1 si P(cancela) pasa de 0.5
print(f"Exactitud: {accuracy_score(yc_prueba, pred_log):.2%}")
confusion_matrix(yc_prueba, pred_log)      # filas: real · columnas: predicho

La **matriz de confusión** cruza lo real (filas) con lo predicho (columnas). En scikit-learn, la
primera fila y la primera columna son el 0 («No canceló»):

| | Predijo «No» | Predijo «Sí» |
|---|---|---|
| **No canceló** | verdaderos negativos (VN) | falsos positivos (FP) |
| **Canceló** | falsos negativos (FN) | verdaderos positivos (VP) |

In [ ]:
# [matriz_grafica]
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(5.6, 4.2))
ConfusionMatrixDisplay.from_predictions(
    yc_prueba, pred_log, display_labels=["No canceló", "Canceló"],
    cmap="Purples", colorbar=False, ax=ax)
ax.set_title("La logística encuentra a menos de la mitad")
ax.set_xlabel("Lo que predijo el modelo")
ax.set_ylabel("Lo que pasó")
plt.show()

Cuatro métricas salen de la matriz:

- **Exactitud** (*accuracy*): de todos, cuántos acertó: $(VP + VN) / n$.
- **Precisión** (*precision*): de los que señaló como «canceló», cuántos cancelaron: $VP / (VP + FP)$.
- **Sensibilidad** (exhaustividad, *recall*): de los que cancelaron, a cuántos encontró: $VP / (VP + FN)$.
- **F1**: la media armónica de la precisión y la sensibilidad, $2PS/(P + S)$; es alta solo si las dos
  lo son.

In [ ]:
# [metricas_c]
from sklearn.metrics import f1_score, precision_score, recall_score
vn, fp, fn, vp = confusion_matrix(yc_prueba, pred_log).ravel()
print(f"VN {vn} · FP {fp} · FN {fn} · VP {vp}")
print(f"Precisión:    {vp / (vp + fp):.3f}",
      f"= {precision_score(yc_prueba, pred_log):.3f}")
print(f"Sensibilidad: {vp / (vp + fn):.3f}",
      f"= {recall_score(yc_prueba, pred_log):.3f}")
print(f"F1:           {f1_score(yc_prueba, pred_log):.3f}")

Como en la regresión, una función junta las métricas de cada modelo. `encontrados` son los verdaderos
positivos: a cuántos de los que cancelaron señaló el modelo. Son las cuentas de la sesión 1 con su nombre:
los «encontrados» son los VP; las «falsas alarmas», los FP, y los que «se escaparon», los FN. «Señalar» a un
cliente es predecir que cancela: en la campaña, es **llamarlo**.

In [ ]:
# [medir_c]
REDONDEO = {"exactitud": 4, "precisión": 3, "sensibilidad": 3, "F1": 3}
def medir_c(modelo):
    """Entrena, decide con el umbral de 0.5 y devuelve las métricas."""
    modelo.fit(Xc_ent, yc_ent)
    pred = modelo.predict(Xc_prueba)
    return {"exactitud": accuracy_score(yc_prueba, pred),
            "precisión": precision_score(yc_prueba, pred, zero_division=0),
            "sensibilidad": recall_score(yc_prueba, pred),
            "F1": f1_score(yc_prueba, pred),
            "encontrados": ((pred == 1) & (yc_prueba == 1)).sum()}

metricas = {"Línea base": medir_c(
                DummyClassifier(strategy="most_frequent")),
            "Logística": medir_c(logistica)}
pd.DataFrame(metricas).T.astype({"encontrados": int}).round(REDONDEO)

## 9 · Árboles: el sobreajuste se ve

Un árbol de clasificación sin límite de profundidad hace lo mismo que en la regresión: parte hasta
memorizar a cada cliente del entrenamiento.

In [ ]:
# [arbol_c]
from sklearn.tree import DecisionTreeClassifier
arbol_c = DecisionTreeClassifier(random_state=2026)   # sin límite
metricas["Árbol sin límite"] = medir_c(arbol_c)
exactitud_entrena = accuracy_score(yc_ent, arbol_c.predict(Xc_ent))
print(f"Exactitud en entrenamiento: {exactitud_entrena:.2%}")
print(f"Exactitud en prueba:        "
      f"{metricas['Árbol sin límite']['exactitud']:.2%}")

Un árbol pequeño, en cambio, se puede **leer**. Con dos niveles de preguntas ya dice dónde está el
riesgo. Cada hoja muestra cuántos clientes del entrenamiento no cancelaron y cuántos sí
(`weights: [no, sí]`). Una regla que se puede leer es una explicación que se puede auditar.

In [ ]:
# [reglas]
from sklearn.tree import export_text
arbol2 = DecisionTreeClassifier(max_depth=2, random_state=2026)
arbol2.fit(Xc_ent, yc_ent)
print(export_text(arbol2, feature_names=list(Xc.columns),
                  show_weights=True))

¿Cómo elige el árbol cada pregunta? Busca la que deja grupos más **puros**: más parecidos entre sí en la
etiqueta (scikit-learn lo mide con un criterio como el índice de Gini). De los 6000 clientes del
entrenamiento cancelaron 1413; con contrato mes a mes, el 44,0 % (871 de 1979), y con permanencia, el 13,5 %
(542 de 4021): ninguna otra pregunta los separa tanto. Por eso un árbol puede hacer una pregunta aunque sus
dos hojas digan lo mismo, como el de la sesión 1: la pregunta sirve si deja grupos más puros, aunque no
cambie la respuesta.

Con límite de profundidad (4 niveles), el árbol ya no memoriza. Y un **bosque aleatorio** de 300
árboles promedia sus votos.

In [ ]:
# [bosque_c]
from sklearn.ensemble import RandomForestClassifier
arbol4 = DecisionTreeClassifier(max_depth=4, random_state=2026)
metricas["Árbol 4 niveles"] = medir_c(arbol4)
bosque_c = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                  random_state=2026, n_jobs=-1)
metricas["Bosque aleatorio"] = medir_c(bosque_c)
pd.DataFrame(metricas).T.astype({"encontrados": int}).round(REDONDEO)

## 10 · Máquina de vectores de soporte (SVM) y XGBoost

Una **SVM** busca la frontera que separa a los que cancelan de los que no con el **margen** más ancho
posible; con el *kernel* RBF (el de `SVC` por defecto), esa frontera puede ser curva. Como KNN, mide
distancias, así que necesita escalar. Mira qué pasa si no se escala.

In [ ]:
# [svm]
from sklearn.svm import SVC
svm_sin = SVC(random_state=2026)                       # sin escalar
svm = make_pipeline(StandardScaler(), SVC(random_state=2026))
sin_escalar = medir_c(svm_sin)
print(f"Sin escalar: exactitud {sin_escalar['exactitud']:.2%},",
      f"encontrados {sin_escalar['encontrados']}")
metricas["SVM"] = medir_c(svm)
print(f"Escalada:    exactitud {metricas['SVM']['exactitud']:.2%},",
      f"encontrados {metricas['SVM']['encontrados']}")

Sin escalar, la SVM hace lo mismo que la línea base: con la factura en pesos (cientos de miles) y el
consumo en GB (cientos), la distancia la deciden casi solo esas dos columnas, que no separan a los que
cancelan; el contrato, la antigüedad o la velocidad casi no cuentan, y el modelo termina diciendo «No» a
todos. Escalada, encuentra a cientos.

**XGBoost** para clasificar funciona como en la regresión: árboles pequeños en secuencia.

In [ ]:
# [xgboost_c]
from xgboost import XGBClassifier
xgb_c = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3,
                      random_state=2026)
metricas["XGBoost"] = medir_c(xgb_c)
tabla_c = pd.DataFrame(metricas).T.astype({"encontrados": int})
tabla_c.round(REDONDEO)

Con el umbral de 0,5, el mejor modelo encuentra a menos de la mitad de los que cancelan. Si la
campaña de retención quiere encontrar a más, hay que **bajar el umbral**: llamar también a clientes con
una probabilidad menor. Se encuentran más, pero se llama a más que no iban a cancelar.

## 11 · El umbral

`predict_proba` da la probabilidad de cada clase; la columna `[:, 1]` es la de cancelar. Con la
logística, probamos cuatro umbrales.

In [ ]:
# [umbral]
prob_log = logistica.predict_proba(Xc_prueba)[:, 1]  # P(cancelar)
filas = {}
for umbral in [0.5, 0.4, 0.3, 0.2]:
    pred = (prob_log >= umbral).astype(int)          # 1 si llega al umbral
    filas[umbral] = {"llamadas": int(pred.sum()),
                     "encontrados": int(pred[yc_prueba == 1].sum()),
                     "precisión": precision_score(yc_prueba, pred),
                     "sensibilidad": recall_score(yc_prueba, pred),
                     "exactitud": accuracy_score(yc_prueba, pred)}
tabla_umbral = pd.DataFrame.from_dict(filas, orient="index")
tabla_umbral.round({"precisión": 3, "sensibilidad": 3, "exactitud": 4})

In [ ]:
# [umbral_grafica]
umbrales = np.arange(0.05, 0.81, 0.01)
precisiones = [precision_score(yc_prueba, (prob_log >= u).astype(int),
                               zero_division=0) for u in umbrales]
sensibilidades = [recall_score(yc_prueba, (prob_log >= u).astype(int))
                  for u in umbrales]
fig, ax = plt.subplots(figsize=(8.4, 3.9))
ax.plot(umbrales, sensibilidades, color="#8A2E9E", linewidth=2.5,
        label="Sensibilidad")
ax.plot(umbrales, precisiones, color="#F46201", linewidth=2.5,
        label="Precisión")
ax.axvline(0.5, color="#566B7F", linestyle="--", linewidth=1)
ax.set_ylim(0, 1)
ax.set_title("Bajar el umbral encuentra más, con menos precisión")
ax.set_xlabel("Umbral de probabilidad para llamar")
ax.set_ylabel("Proporción")
ax.legend(frameon=False)
plt.show()

## 12 · Validación cruzada, sesgo y varianza

Una sola partición puede tener suerte. La **validación cruzada** parte el **entrenamiento** en 5
pliegues; entrena 5 veces, cada vez con 4 pliegues, y mide en el que dejó por fuera. Da 5 puntajes:
su promedio estima el desempeño y su dispersión, qué tanto depende de los datos. La prueba sigue
guardada para el final. `StratifiedKFold` conserva la tasa de cancelación en cada pliegue.

In [ ]:
# [validacion_cruzada]
from sklearn.model_selection import StratifiedKFold, cross_val_score
pliegues = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
f1_xgb = cross_val_score(xgb_c, Xc_ent, yc_ent, cv=pliegues,
                         scoring="f1")
print("F1 de cada pliegue:", np.round(f1_xgb, 3))
print(f"F1 medio: {f1_xgb.mean():.3f} · desviación: {f1_xgb.std():.3f}")

Con la validación cruzada se ve el **sesgo** y la **varianza** de un modelo. Un árbol con pocos niveles
es demasiado simple: se equivoca parecido en entrenamiento y en validación (**sesgo**, subajuste). Uno
con muchos acierta casi todo en entrenamiento y falla en validación (**varianza**, sobreajuste). Entre
los dos está la profundidad que mejor generaliza.

In [ ]:
# [curva_profundidad]
from sklearn.model_selection import cross_validate
profundidades = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20]
entrena, valida = [], []
for p in profundidades:
    cv = cross_validate(DecisionTreeClassifier(max_depth=p,
                                               random_state=2026),
                        Xc_ent, yc_ent, cv=pliegues,
                        scoring="accuracy", return_train_score=True)
    entrena.append(cv["train_score"].mean())
    valida.append(cv["test_score"].mean())
fig, ax = plt.subplots(figsize=(8.4, 3.9))
ax.plot(profundidades, entrena, "o-", color="#8A2E9E", linewidth=2.5,
        label="Entrenamiento")
ax.plot(profundidades, valida, "o-", color="#F46201", linewidth=2.5,
        label="Validación cruzada")
ax.set_title("Más niveles: el árbol memoriza y generaliza peor")
ax.set_xticks(profundidades)
ax.set_xlabel("Profundidad máxima del árbol (niveles)")
ax.set_ylabel("Exactitud")
ax.legend(frameon=False)
plt.show()

## 13 · ¿Qué modelo le sirve a la gerente?

La gerente de experiencia del cliente no decide con el umbral de 0,5: decide **a cuántos llamar**. Su
meta es encontrar a 7 de cada 10 de los que van a cancelar. Ordenamos a los clientes de prueba de
mayor a menor riesgo según cada modelo y contamos cuántas llamadas hacen falta para llegar a la meta.
Un modelo que ordena mejor necesita menos llamadas. La SVM no da probabilidades, pero su
`decision_function` ordena igual: a más alta, más riesgo. Los clientes con el mismo riesgo se llaman
todos juntos (el árbol de 4 niveles solo da unas pocas probabilidades distintas).

In [ ]:
# [llamadas]
def llamadas_para(riesgo, meta=0.70):
    """Llamadas, de mayor a menor riesgo, para encontrar a la meta."""
    y = yc_prueba.to_numpy()
    for corte in np.sort(np.unique(riesgo))[::-1]:   # de mayor a menor
        llamados = riesgo >= corte          # los empatados, todos juntos
        if y[llamados].sum() >= meta * y.sum():
            return int(llamados.sum())

riesgos = {"Logística": prob_log,
           "Árbol 4 niveles": arbol4.predict_proba(Xc_prueba)[:, 1],
           "Bosque aleatorio": bosque_c.predict_proba(Xc_prueba)[:, 1],
           "SVM": svm.decision_function(Xc_prueba),
           "XGBoost": xgb_c.predict_proba(Xc_prueba)[:, 1]}
tabla_llamadas = pd.DataFrame(
    {"llamadas para 7 de cada 10":
         {m: llamadas_para(r) for m, r in riesgos.items()}})
tabla_llamadas["exactitud con 0.5"] = tabla_c["exactitud"].round(4)
tabla_llamadas.sort_values("llamadas para 7 de cada 10")

Si llamara al azar, para encontrar a 7 de cada 10 tendría que llamar a 7 de cada 10 clientes. XGBoost
llega a la meta con menos llamadas que la logística y la SVM; el bosque queda casi empatado, y con una
sola partición esa diferencia no se puede dar por segura. Pero no es lo único que cuenta: la gerente también
tiene que poder **explicar** por qué se llama a un cliente, y el árbol pequeño y la logística se
explican mejor. Esa es la «explicación auditable» del proyecto final.

---
## Ejercicio 2 · La matriz de confusión del bosque

Con el bosque aleatorio ya entrenado (`bosque_c`, en la celda [bosque_c]), predice la prueba con el
umbral de 0,5 (`predict`), calcula su matriz de confusión y saca de ella los cuatro números. Pista: las
celdas [logistica] y [metricas_c]: el orden de `.ravel()` es VN, FP, FN, VP.

Después responde abajo: el bosque acierta un poco más que la logística; con sus VP y FP, ¿cuál le sirve
más a la gerente?

In [ ]:
# [ej2]
pred_bosque = ...                          # predice la prueba
matriz = ...                               # la matriz de confusión
vn_b = fp_b = fn_b = vp_b = ...   # sus cuatro números: cambia toda la línea
revisar_matriz(vp=vp_b, fp=fp_b, fn=fn_b, vn=vn_b)

## Ejercicio 3 · El umbral de la gerente

La gerente quiere encontrar **al menos a 7 de cada 10** de los que cancelan (sensibilidad de 0,70 o
más), con el menor número de llamadas. Con las probabilidades de **XGBoost**, completa el ciclo: para
cada umbral, la sensibilidad, las llamadas (cuántos clientes quedan por encima del umbral) y la
exactitud. Elige el umbral **más alto** que cumple la meta, y entrégalo con sus llamadas. Pista: la
celda [umbral].

Para simplificar, aquí elegimos el umbral mirando la prueba. En el proyecto, elíjanlo con validación
cruzada sobre el entrenamiento y midan la prueba una sola vez, al final.

Después responde abajo: la gerente solo puede pagar 500 llamadas. ¿Qué umbral de la lista usarías?
¿Cuántos encontrarías? ¿Sigues cumpliendo la meta? Y una más: la lista salta de 0,05 en 0,05; prueba
con pasos de 0,01 cerca del umbral que elegiste: ¿cuántas llamadas te ahorras?

In [ ]:
# [ej3]
prob_xgb = xgb_c.predict_proba(Xc_prueba)[:, 1]      # P(cancelar)
for umbral in [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20, 0.15]:
    pred = (prob_xgb >= umbral).astype(int)
    sensibilidad = ...                     # recall_score(...)
    llamadas = ...                         # cuántos 1 hay en pred
    exactitud = ...                        # accuracy_score(...)
    print(f"umbral {umbral:.2f} ·", "sensibilidad", sensibilidad,
          "· llamadas", llamadas, "· exactitud", exactitud)
umbral_elegido = ...                       # el más alto que cumple
llamadas_elegido = ...                     # sus llamadas
revisar_umbral(umbral=umbral_elegido, llamadas=llamadas_elegido)

## Ejercicio 4 (opcional) · La profundidad que elige la validación cruzada

Para cada profundidad de 1 a 10, calcula el **F1 medio** de un árbol de clasificación en la validación
cruzada de 5 pliegues (`pliegues`, sobre el **entrenamiento**). Elige la profundidad con el F1 medio
más alto y entrégala con ese F1. Pista: la celda [validacion_cruzada].

Después responde abajo: ¿por qué no se elige la profundidad con la prueba?

In [ ]:
# [ej4]
for profundidad in range(1, 11):
    arbol_p = DecisionTreeClassifier(max_depth=profundidad,
                                     random_state=2026)
    f1_medio = ...                         # promedio de cross_val_score
    print(profundidad, f1_medio)
mejor_profundidad = ...
mejor_f1 = ...
revisar_profundidad(profundidad=mejor_profundidad, f1_medio=mejor_f1)

---
# Parte 3 · Nube: ¿quién puede ver qué en Azure?

## Ejercicio de diseño · La matriz de roles de Yarumo

Este ejercicio no lleva código: se hace en las salas, con las diapositivas de la sesión. Yarumo guarda
en Azure, en el grupo de recursos `rg-yarumo-churn`, una cuenta de almacenamiento con tres contenedores:

- `datos-clientes`: la tabla seudonimizada, que sigue teniendo datos personales. La carga cada mes un
  proceso automático desde el CRM (el sistema de gestión de clientes).
- `modelos`: el modelo entrenado.
- `informes`: los resultados de cada campaña, sin datos de clientes uno a uno. Los escribe la aplicación.

Los grupos: **Analítica** (tu equipo) entrena y evalúa los modelos; **TI** administra la infraestructura
(crea, configura y borra recursos) y asigna los accesos; la **gerencia** lee los resultados; el
**proveedor externo** es una consultora que revisa el modelo durante un mes, y **la aplicación que
predice** calcula el riesgo de cada cliente con su identidad administrada.

Para cada grupo, escribe el **rol integrado** de Azure y el **ámbito** más estrecho que le basta, o
«sin acceso», y justifica con el privilegio mínimo. *Doble clic para escribir aquí*; si trabajan en sala,
pueden escribir una línea por grupo en el chat.

| Grupo | `datos-clientes` | `modelos` | `informes` | Por qué |
|---|---|---|---|---|
| Analítica | … | … | … | … |
| TI | … | … | … | … |
| Gerencia | … | … | … | … |
| Proveedor externo | … | … | … | … |
| La aplicación que predice (identidad administrada) | … | … | … | … |

---
### Tus respuestas

*Doble clic para escribir aquí.*

1. Ejercicio 1: el árbol de 6 niveles tiene un R² de … en entrenamiento y de … en prueba (el sin límite:
   1 y …). ¿Se sobreajusta menos? ¿Cómo lo sabes? ¿Lo elegirías antes que el bosque solo porque la
   diferencia entre entrenamiento y prueba es menor? ¿Qué mirarías? …
2. Ejercicio 2: el bosque encontró a … y señaló a … que no cancelaron; la logística, a 186 y a 98. ¿Cuál le
   sirve más a la gerente y por qué? ¿Cambia tu respuesta si cada llamada es muy cara? …
3. Ejercicio 3: con el umbral de …, XGBoost encuentra a … con … llamadas, y la exactitud pasa de … a … Con
   500 llamadas, ¿qué umbral usarías y cumplirías la meta? Con pasos de 0,01, ¿cuántas llamadas ahorras? …
4. Ejercicio 4 (opcional): la validación cruzada elige … niveles. ¿Por qué no se elige con la prueba? …

### Autoevaluación (respóndela en el cierre de la sesión)

¿Verdadero o falso? Explica cada una.

1. «Si un modelo tiene una exactitud del 76 %, encuentra a 3 de cada 4 de los que cancelan». …
2. «Si un árbol tiene un R² de 1 en entrenamiento, con clientes nuevos tendrá un R² parecido». …
3. «Bajar el umbral sube la sensibilidad y suele bajar la precisión». …
4. «Un proveedor con el rol Lector de datos de blobs en el contenedor `modelos` no puede leer
   `datos-clientes`». …

---
## La respuesta a la gerencia

**¿Cuántos GB consumirá cada cliente?** XGBoost y el bosque aleatorio se equivocan en promedio unos
70 GB por cliente, frente a unos 170 de la línea base, porque combinan columnas, como la edad con el
número de usuarios, y la regresión lineal solo las suma.

**¿Quién va a cancelar?** Con el umbral de 0,5, el mejor modelo encuentra a menos de la mitad de los
que cancelan. Para encontrar a 7 de cada 10, XGBoost necesita menos llamadas que los demás modelos.
La exactitud no sirve para elegir: la línea base acierta el 76 % y no encuentra a nadie. Y un modelo
que se usa para llamar a clientes se tiene que poder explicar.

**¿Quién puede ver qué en Azure?** Cada grupo recibe un rol integrado en el ámbito más estrecho que le
basta: los datos personales, solo para Analítica y para la aplicación que predice; TI administra los
recursos sin leer el contenido, y Azure Policy fija lo que nadie puede saltarse.